In [1]:
import sys

print(sys.executable)
print(sys.version)

C:\Users\DARSHAN\AppData\Local\Programs\Python\Python312\python.exe
3.12.8 (tags/v3.12.8:2dc476b, Dec  3 2024, 19:30:04) [MSC v.1942 64 bit (AMD64)]


In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

In [3]:
iris = load_iris()
X = StandardScaler().fit_transform(iris.data)

In [4]:
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["species"] = iris.target_names[iris.target]

In [5]:
wcss = [KMeans(n_clusters=k, n_init=10, random_state=42).fit(X).inertia_
for k in range(1, 11)]
plt.plot(range(1, 11), wcss, marker="o")
plt.title("Elbow Method"); plt.xlabel("K"); plt.ylabel("WCSS")
plt.savefig("elbow_method_short.png")
plt.close()

In [6]:
k = 3
kmeans_labels = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
df["kmeans_cluster"] = kmeans_labels
print("K-Means silhouette:", round(silhouette_score(X, kmeans_labels), 3))

K-Means silhouette: 0.46


In [7]:
linked = linkage(X, method="ward")
dendrogram(linked, no_labels=True)
plt.title("Dendrogram")
plt.savefig("dendrogram_short.png")
plt.close()

In [8]:
hier_labels = AgglomerativeClustering(
    n_clusters=k,
    linkage="ward"
).fit_predict(X)
df["hier_cluster"] = hier_labels
print("Hierarchical silhouette:", round(silhouette_score(X, hier_labels), 3))
print(pd.crosstab(df["species"], df["kmeans_cluster"]))

Hierarchical silhouette: 0.447
kmeans_cluster   0   1   2
species                   
setosa           0  50   0
versicolor      39   0  11
virginica       14   0  36


In [9]:
print("K-Means Cluster Means:")
print(df.groupby("kmeans_cluster")[iris.feature_names].mean().round(2))

print("\nK-Means Cross Tabulation:")
print(pd.crosstab(df["species"], df["kmeans_cluster"]))

print("\nHierarchical Cross Tabulation:")
print(pd.crosstab(df["species"], df["hier_cluster"]))

K-Means Cluster Means:
                sepal length (cm)  sepal width (cm)  petal length (cm)  \
kmeans_cluster                                                           
0                            5.80              2.67               4.37   
1                            5.01              3.43               1.46   
2                            6.78              3.10               5.51   

                petal width (cm)  
kmeans_cluster                    
0                           1.41  
1                           0.25  
2                           1.97  

K-Means Cross Tabulation:
kmeans_cluster   0   1   2
species                   
setosa           0  50   0
versicolor      39   0  11
virginica       14   0  36

Hierarchical Cross Tabulation:
hier_cluster   0   1   2
species                 
setosa         0  49   1
versicolor    23   0  27
virginica     48   0   2
